In [ ]:
# 1. INSTALL AND IMPORT
!pip install numpy pandas networkx plotly
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# 2. CORE CLASSES (Agent & Simulation)
class MarketAgent:
    def __init__(self, agent_id, personality='random', initial_money=1000):
        self.id, self.personality, self.money = agent_id, personality, initial_money
        self.stocks, self.history, self.neighbors = 0, [], []
        self.thresholds = {'cautious': 0.7, 'aggressive': 0.3, 'random': 0.5, 'contrarian': 0.4}

    def decide_action(self, current_price, neighbor_actions):
        if not neighbor_actions: return 'hold'
        buy_pressure = neighbor_actions.count('buy') / len(neighbor_actions)
        t = self.thresholds[self.personality]
        if self.personality == 'contrarian':
            return 'sell' if buy_pressure > t and self.stocks > 0 else 'buy' if buy_pressure < (1-t) and self.money >= current_price else 'hold'
        return 'buy' if buy_pressure > t and self.money >= current_price else 'sell' if buy_pressure < (1-t) and self.stocks > 0 else 'hold'

    def take_action(self, action, current_price):
        if action == 'buy': self.stocks += 1; self.money -= current_price
        elif action == 'sell': self.stocks -= 1; self.money += current_price
        self.history.append({'total_value': self.money + (self.stocks * current_price)})

class MarketSimulation:
    def __init__(self, num_agents=100):
        self.agents = [MarketAgent(i, ['cautious', 'aggressive', 'random', 'contrarian'][i % 4]) for i in range(num_agents)]
        self.network = nx.barabasi_albert_graph(num_agents, 3)
        for i in range(num_agents): self.agents[i].neighbors = list(self.network.neighbors(i))
        self.price, self.price_history, self.volume_history = 100, [100], [0]

    def step(self):
        actions = []
        for agent in self.agents:
            action = agent.decide_action(self.price, [actions[n] for n in agent.neighbors if n < len(actions)])
            actions.append(action)
            agent.take_action(action, self.price)
        change = (actions.count('buy') - actions.count('sell')) / len(self.agents)
        self.price *= (1 + change * 0.1)
        self.price_history.append(self.price)
        self.volume_history.append(actions.count('buy') + actions.count('sell'))

    def run(self, steps=100):
        for _ in range(steps): self.step()

In [ ]:
# 3. VISUALIZATION FUNCTIONS
def plot_results(sim):
    fig = make_subplots(rows=2, cols=1, subplot_titles=('Market Price', 'Trading Volume'))
    fig.add_trace(go.Scatter(y=sim.price_history, name="Price"), row=1, col=1)
    fig.add_trace(go.Bar(y=sim.volume_history, name="Volume"), row=2, col=1)
    fig.show()

def plot_net(sim):
    pos = nx.spring_layout(sim.network)
    edge_x, edge_y = [], []
    for edge in sim.network.edges():
        x0, y0 = pos[edge[0]]; x1, y1 = pos[edge[1]]
        edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

    node_x = [pos[i][0] for i in range(len(sim.agents))]
    node_y = [pos[i][1] for i in range(len(sim.agents))]
    wealth = [a.money + (a.stocks * sim.price) for a in sim.agents]

    fig = go.Figure(data=[go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), mode='lines'),
                         go.Scatter(x=node_x, y=node_y, mode='markers', marker=dict(size=10, color=wealth, colorscale='YlOrRd', showscale=True))])
    fig.update_layout(title="Wealth Distribution Network")
    fig.show()

In [ ]:
# 4. RUN SIMULATION
print("Running Market Simulation...")
sim = MarketSimulation(num_agents=100)
sim.run(steps=100)
plot_results(sim)
plot_net(sim)